# Train the custom model `custom-shloka-mt`

Fine-tunes AI4Bharat **IndicTrans2** (Indic→English) on shloka/classical material and saves it to
`models/custom/custom-shloka-mt_<date>_<time>/`, where the translator picks it up as **custom-shloka-mt**.

* T4: `ai4bharat/indictrans2-indic-en-dist-200M`, full fine-tune (fp16 + gradient accumulation).
* L4 / A100: `ai4bharat/indictrans2-indic-en-1B` with LoRA (`use_lora=True`).
* Checkpoints go to `training.checkpoint_dir` (default `models/checkpoints` inside the repo, i.e. on Drive) every `save_steps`; re-running the training cell after a disconnect resumes.
* Leakage guard: pairs matching any shloka in `reference/` or `Input_Folder/` are removed.

## 1 · GPU check and repository location (no hard-coded paths)
Same discovery logic as notebook 01: the repo is found by its contents in the current folder or anywhere on your Drive; set `REPO_DIR_OVERRIDE` to skip the search.

In [ ]:
!nvidia-smi || echo '⚠ No GPU — training needs a GPU runtime.'

# Locate the repository WITHOUT hard-coded paths:
#   1. REPO_DIR_OVERRIDE below or the SHLOKA_REPO_DIR environment variable (optional),
#   2. the current folder and its parents (works locally / in Jupyter),
#   3. Colab: mount Google Drive (re-using an existing mount) and search it for the folder that
#      contains shloka_translator/__init__.py + config/config.yaml — whatever its name or location.
import os
import sys
from pathlib import Path

REPO_DIR_OVERRIDE = ""  #@param {type:"string"}
MAX_SEARCH_DEPTH = 6    #@param {type:"integer"}
IN_COLAB = "google.colab" in sys.modules


def is_repo(path: Path) -> bool:
    return (path / "shloka_translator" / "__init__.py").is_file() and (path / "config" / "config.yaml").is_file()


def existing_drive_mount() -> Path | None:
    """Return the mount point of an already-mounted Google Drive, if any."""
    import psutil
    for part in psutil.disk_partitions(all=True):
        mp = Path(part.mountpoint)
        if mp.name == "drive" and ("fuse" in part.fstype.lower() or "drive" in part.device.lower()):
            return mp
    return None


def search(root: Path, max_depth: int) -> list[Path]:
    """Breadth-first, depth-limited search for repository folders under ``root``."""
    hits: list[Path] = []
    for depth in range(0, max_depth + 1):
        pattern = "/".join(["*"] * depth + ["shloka_translator", "__init__.py"])
        hits = [p.parent.parent for p in root.glob(pattern) if ".Trash" not in p.parts and is_repo(p.parent.parent)]
        if hits:
            break
    return hits


REPO_DIR = None
override = REPO_DIR_OVERRIDE.strip() or os.environ.get("SHLOKA_REPO_DIR", "").strip()
if override:
    REPO_DIR = Path(override).expanduser().resolve()
    if not is_repo(REPO_DIR):
        raise FileNotFoundError(f"{REPO_DIR} does not contain shloka_translator/ and config/config.yaml")
else:
    REPO_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if is_repo(p)), None)

if REPO_DIR is None and IN_COLAB:
    from google.colab import drive
    mount_point = existing_drive_mount()
    if mount_point is None:
        mount_point = Path.cwd() / "drive"      # Colab's default working folder + "drive"
        drive.mount(str(mount_point))
    print(f"Searching {mount_point} for the repository (depth ≤ {MAX_SEARCH_DEPTH})…")
    candidates = search(mount_point, MAX_SEARCH_DEPTH)
    if not candidates:
        raise FileNotFoundError("Repository not found on Google Drive. Upload the unzipped folder anywhere in "
                                "your Drive (or set REPO_DIR_OVERRIDE) and re-run this cell.")
    if len(candidates) > 1:
        print("Several copies found (using the most recently modified):", *candidates, sep="\n  ")
    REPO_DIR = max(candidates, key=lambda p: (p / "config" / "config.yaml").stat().st_mtime)

if REPO_DIR is None:
    raise FileNotFoundError("Repository not found. Start Jupyter inside the repo or set REPO_DIR_OVERRIDE.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("REPO_DIR =", REPO_DIR)

## 2 · Install training requirements

In [ ]:
!python scripts/install_requirements.py requirements-train.txt
import importlib; importlib.invalidate_caches()

## 3 · Secrets
`HF_TOKEN` (🔑 sidebar) is required: the IndicTrans2 checkpoints and BPCC are **gated** — accept their
terms on Hugging Face first.

In [ ]:
from shloka_translator.secrets import get_secret
from huggingface_hub import login
HF_TOKEN = get_secret("HF_TOKEN")
assert HF_TOKEN, "Add HF_TOKEN in Colab Secrets (🔑) and enable notebook access."
login(token=HF_TOKEN, add_to_git_credential=False)

## 4 · Settings
`QUICK_RUN=True` trains on a small subset for a few hundred steps (a smoke test, ~10–20 min on a T4).
Set it to `False` for the full run (all of Itihāsa + BPCC sample; hours).

In [ ]:
from shloka_translator.config import load_config
QUICK_RUN = False
USE_LORA_1B = False            # True on L4/A100: indictrans2-indic-en-1B + LoRA

overrides = {"training": {}}
if USE_LORA_1B:
    overrides["training"].update(base_model="ai4bharat/indictrans2-indic-en-1B", use_lora=True,
                                 per_device_batch_size=8)
if QUICK_RUN:
    overrides["training"].update(use_bpcc=False, num_epochs=1, warmup_steps=50, save_steps=100, eval_steps=100)
cfg = load_config(overrides=overrides)
SUBSET = 3000 if QUICK_RUN else None
MAX_STEPS = 300 if QUICK_RUN else None
print(cfg.training.model_dump())

## 5 · Inspect the training data (sources, cleaning, leakage guard)
`rahular/itihasa` is published with a *loading script* (`itihasa.py`). `datasets` ≥ 4.0 (Colab's current
version) refuses to run such scripts, so the loader automatically falls back to the Hub's auto-converted
Parquet files — the `via …` note below shows which route was used. The cell stops with a clear message if
no training pairs were collected, instead of failing later inside `train()`.

In [ ]:
from shloka_translator.training.data_prep import prepare_training_data
preview = prepare_training_data(cfg, HF_TOKEN, subset=SUBSET)
for s in preview.sources:
    print(f"{s.status:8s} {s.pairs:7d}  {s.name}  ({s.licence}) {s.note[:100]}")
print(preview.stats)
if not preview.train:
    raise SystemExit("No training pairs collected — read the skipped/notes lines above "
                     "(network, HF_TOKEN, or add files to Training_Data/).")
print(f"✓ {len(preview.train)} training pairs ready")


## 6 · Train (resumes automatically from the latest Drive checkpoint)

In [ ]:
from shloka_translator.training.train import train
out_dir = train(cfg, HF_TOKEN, subset=SUBSET, max_steps=MAX_STEPS)
print("Saved:", out_dir)

## 7 · Evaluation: base vs fine-tuned (sacreBLEU, chrF++)
Scores are on the held-out test split. They say nothing about GPT/LLM backends unless those are
scored on the same split.

In [ ]:
import json, pandas as pd
m = json.loads((out_dir / "metrics.json").read_text())
cols = ["Lang", "n", "BLEU base", "BLEU tuned", "ΔBLEU", "chrF++ base", "chrF++ tuned", "ΔchrF++"]
display(pd.DataFrame(m["comparison"], columns=cols) if m["comparison"] else m["fine_tuned"])
print((out_dir / "model_card.md").read_text()[:2000])

## 8 · Check that the translator sees the new model

In [ ]:
from shloka_translator.models.custom_backend import CustomBackend
print(CustomBackend("custom-shloka-mt", load_config()).is_available())